# AI Agent Governance Framework - Demo Analysis

This notebook demonstrates the governance framework with visualizations and analysis.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
import json
from framework.agent_registry import AgentRegistry
from framework.evaluator import AgentEvaluator
from framework.governance import GovernanceEngine
from framework.audit_logger import AuditLogger

## 1. Load and Explore Agents

In [ ]:
# Load dummy data
agents_df = pd.read_csv('../dummy_data/agents.csv')
agents_df.head()

In [ ]:
# Risk level distribution
risk_counts = agents_df['risk_level'].value_counts()
plt.figure(figsize=(8, 5))
risk_counts.plot(kind='bar', color=['green', 'orange', 'red'])
plt.title('Agent Distribution by Risk Level')
plt.xlabel('Risk Level')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 2. Register Agents

In [ ]:
registry = AgentRegistry(storage_path='../registry_db.json')
audit = AuditLogger(log_file='../audit_log.json')

for _, row in agents_df.iterrows():
    tools = row['authorized_tools'].split(',') if isinstance(row['authorized_tools'], str) else []
    registry.register_agent(
        agent_id=row['agent_id'],
        name=row['agent_name'],
        agent_type=row['agent_type'],
        purpose=row['purpose'],
        tools=tools,
        risk_level=row['risk_level']
    )

print(f"Registered {len(registry.list_agents())} agents")

## 3. Evaluate Sample Agent

In [ ]:
evaluator = AgentEvaluator()

# Get first high-risk agent
agent = registry.get_agents_by_risk('high')[0]
print(f"Evaluating: {agent['name']} ({agent['id']})")

In [ ]:
# Universal metrics
universal_scores = {
    'reliability': 0.91,
    'transparency': 0.88,
    'accountability': 0.85,
    'safety_score': 0.92
}

universal_result = evaluator.evaluate_universal(universal_scores)

# Plot universal metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

metrics = [d['metric'] for d in universal_result['details']]
values = [d['value'] for d in universal_result['details']]
thresholds = [d['threshold'] for d in universal_result['details']]

x = range(len(metrics))
ax1.bar(x, values, label='Actual', alpha=0.7)
ax1.bar(x, thresholds, label='Threshold', alpha=0.5)
ax1.set_xticks(x)
ax1.set_xticklabels(metrics, rotation=45, ha='right')
ax1.set_ylabel('Score')
ax1.set_title('Universal Metrics vs Thresholds')
ax1.legend()
ax1.set_ylim(0, 1)

# Grade distribution
ax2.pie([1], labels=[universal_result['grade']], autopct='%1.0f%%', 
        colors=['green' if universal_result['passed'] else 'red'])
ax2.set_title(f"Overall Grade: {universal_result['grade']}")

plt.tight_layout()
plt.show()

print(f"Overall Score: {universal_result['overall_score']}")
print(f"Grade: {universal_result['grade']}")

## 4. Governance Compliance Check

In [ ]:
governance = GovernanceEngine(audit)

# Test with insufficient controls
context_fail = {
    'logging': True,
    'approval': False,
    'authorization_count': 1,
    'kill_switch': False
}

result_fail = governance.evaluate(agent, context_fail)

# Test with full controls
context_pass = {
    'logging': True,
    'approval': True,
    'authorization_count': 2,
    'kill_switch': True,
    'compliance_review': True
}

result_pass = governance.evaluate(agent, context_pass)

In [ ]:
# Visualize governance results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (result, title) in enumerate([(result_fail, 'Insufficient Controls'), 
                                        (result_pass, 'Full Controls')]):
    ax = axes[idx]
    
    rules = [r.rule_id for r in result.rule_results]
    statuses = [1 if r.status.value == 'pass' else 0 for r in result.rule_results]
    colors = ['green' if s == 1 else 'red' for s in statuses]
    
    ax.barh(rules, statuses, color=colors, alpha=0.7)
    ax.set_xlim(0, 1.2)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Fail', 'Pass'])
    ax.set_title(f"{title}\nStatus: {result.overall_status.value.upper()}")
    
plt.tight_layout()
plt.show()

## 5. Audit Log Analysis

In [ ]:
# Load audit log
with open('../audit_log.json', 'r') as f:
    audit_logs = json.load(f)

print(f"Total audit events: {len(audit_logs)}")

# Event type distribution
event_types = {}
for log in audit_logs:
    event_type = log.get('event_type', 'unknown')
    event_types[event_type] = event_types.get(event_type, 0) + 1

plt.figure(figsize=(8, 5))
plt.bar(event_types.keys(), event_types.values())
plt.title('Audit Events by Type')
plt.xlabel('Event Type')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Verify integrity
is_valid = audit.verify_integrity()
print(f"Audit log integrity: {'VALID' if is_valid else 'CORRUPTED'}")

## 6. Comprehensive Evaluation Summary

In [ ]:
# Run comprehensive evaluation
workflow_scores = {
    'execution_speed': 0.96,
    'slippage_control': 0.91,
    'compliance_adherence': 0.97,
    'risk_management': 0.94
}

comprehensive = evaluator.comprehensive_evaluation(
    agent=agent,
    universal_scores=universal_scores,
    workflow_type='trading',
    workflow_scores=workflow_scores
)

print(f"Agent: {comprehensive.agent_name}")
print(f"Combined Score: {comprehensive.overall_score}")
print(f"Final Status: {comprehensive.status.upper()}")
print(f"Timestamp: {comprehensive.timestamp}")